# Polymarket — Closed sports-market dataset

## Goal

Create a dataset of **closed Polymarket sports-market candidates**.

For this stage, we do **not**:

- determine the resolved outcome;
- manually decide whether a market is an actual sports match;
- fetch historical CLOB prices;
- restrict to binary markets.

Those steps come later.

### Dataset definition

Instead of downloading every closed Polymarket market and trying to identify sports locally, we query Gamma directly using Polymarket's own:

- **Sports tag** (`tag_id = 1`)
- `related_tags = true`

Gamma therefore expands the Sports tag server-side to markets associated with related sports tags.

This gives us a much smaller and more defensible candidate universe for later manual annotation.


In [1]:
import time
from pathlib import Path

import pandas as pd
import requests

GAMMA_BASE_URL = "https://gamma-api.polymarket.com"

# Polymarket's official Sports tag
SPORTS_TAG_ID = 1

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## 1. Verify the Sports tag

Before collecting markets, verify what `SPORTS_TAG_ID` refers to.

This is mainly documentation / a sanity check. We do **not** manually define all related sports tags in Python; Gamma handles that expansion when we later pass `related_tags=true`.


In [2]:
response = requests.get(
    f"{GAMMA_BASE_URL}/tags/{SPORTS_TAG_ID}",
    timeout=30,
)

response.raise_for_status()

sports_tag = response.json()
sports_tag

{'$schema': 'https://gamma-api.polymarket.com/schemas/Tag.json',
 'id': '1',
 'label': 'Sports',
 'slug': 'sports',
 'forceShow': False,
 'publishedAt': '2023-10-24 22:37:50.296+00',
 'updatedBy': 15,
 'createdAt': '2023-10-24T22:37:50.31Z',
 'updatedAt': '2026-04-17T20:47:16.461062Z',
 'forceHide': True}

In [3]:
print("Tag ID:", sports_tag.get("id"))
print("Label:", sports_tag.get("label"))
print("Slug:", sports_tag.get("slug"))


Tag ID: 1
Label: Sports
Slug: sports


In [4]:
response = requests.get(
    f"{GAMMA_BASE_URL}/tags/{SPORTS_TAG_ID}/related-tags/tags"
)

response.raise_for_status()

related_sports_tags = response.json()
related_sports_tags

[{'id': '450',
  'label': 'NFL (All)',
  'slug': 'nfl',
  'forceShow': False,
  'publishedAt': '2023-11-02 23:11:18.762+00',
  'createdAt': '2023-11-02T23:11:18.767Z',
  'updatedAt': '2026-09-18T16:00:07.209724Z',
  'activeEventsCount': 566},
 {'id': '100350',
  'label': 'Soccer',
  'slug': 'soccer',
  'forceShow': False,
  'createdAt': '2024-08-21T16:31:18.953213Z',
  'updatedAt': '2026-04-17T20:33:09.955379Z',
  'activeEventsCount': 10160},
 {'id': '306',
  'label': 'EPL',
  'slug': 'epl',
  'forceShow': False,
  'publishedAt': '2023-11-02 21:53:01.244+00',
  'createdAt': '2023-11-02T21:53:01.251Z',
  'updatedAt': '2026-04-17T20:31:50.149157Z',
  'activeEventsCount': 204},
 {'id': '100381',
  'label': 'MLB',
  'slug': 'mlb',
  'createdAt': '2024-08-27T16:42:20.841409Z',
  'updatedAt': '2026-04-17T20:23:34.251722Z',
  'activeEventsCount': 117},
 {'id': '1234',
  'label': 'Champions League',
  'slug': 'champions-league',
  'forceShow': False,
  'publishedAt': '2024-02-06 20:04:23.286+0

## 2. Fetch closed sports markets

We use Gamma's **keyset pagination** endpoint because ordinary offset pagination does not scale to large result sets.

The important filters are:

```python
closed=true
tag_id=SPORTS_TAG_ID
related_tags=true
```

So we request only closed markets connected to Polymarket's Sports taxonomy.

`next_cursor` from each response is passed back as `after_cursor` for the next page.


In [5]:
def fetch_closed_sports_markets(
    page_size=100,
    max_pages=None,
    sleep_seconds=0.1,
):
    all_markets = []
    after_cursor = None
    page = 0

    while True:
        params = {
            "closed": "true",
            "tag_id": SPORTS_TAG_ID,
            "related_tags": "true",
            "limit": page_size,
        }

        if after_cursor is not None:
            params["after_cursor"] = after_cursor

        response = requests.get(
            f"{GAMMA_BASE_URL}/markets/keyset",
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        payload = response.json()

        batch = payload.get("markets", [])
        next_cursor = payload.get("next_cursor")

        if not batch:
            break

        all_markets.extend(batch)

        print(
            f"Page {page + 1}: {len(batch):>3} markets "
            f"| total sports candidates = {len(all_markets)}"
        )

        page += 1

        if max_pages is not None and page >= max_pages:
            break

        if not next_cursor:
            break

        after_cursor = next_cursor
        time.sleep(sleep_seconds)

    return all_markets


### Run the collection

For the final dataset, `max_pages=None` is fine here because the filtering happens **inside Gamma**, before the data are returned.

If you want a quick test first, temporarily use something like `max_pages=3`.


In [6]:
raw_markets = fetch_closed_sports_markets(
    page_size=100,
    max_pages=None,
)

markets_df = pd.DataFrame(raw_markets)

print("\nTotal closed sports candidates:", len(markets_df))
print("Columns:", len(markets_df.columns))

markets_df.head()


Page 1: 100 markets | total sports candidates = 100
Page 2: 100 markets | total sports candidates = 200
Page 3: 100 markets | total sports candidates = 300
Page 4: 100 markets | total sports candidates = 400
Page 5: 100 markets | total sports candidates = 500
Page 6: 100 markets | total sports candidates = 600
Page 7: 100 markets | total sports candidates = 700
Page 8: 100 markets | total sports candidates = 800
Page 9: 100 markets | total sports candidates = 900
Page 10: 100 markets | total sports candidates = 1000
Page 11: 100 markets | total sports candidates = 1100
Page 12: 100 markets | total sports candidates = 1200
Page 13: 100 markets | total sports candidates = 1300
Page 14: 100 markets | total sports candidates = 1400
Page 15: 100 markets | total sports candidates = 1500
Page 16: 100 markets | total sports candidates = 1600
Page 17: 100 markets | total sports candidates = 1700
Page 18: 100 markets | total sports candidates = 1800
Page 19: 100 markets | total sports candidates

,id,question,conditionId,slug,resolutionSource,endDate,category,liquidity,startDate,fee,...,sportsMarketType,acceptingOrdersTimestamp,gameId,line,deployingTimestamp,feeSchedule,eventStartTime,groupItemRange,positionIds,marketMetadata
0,246670,Formula 1: Which team will collect more points...,0x6818e44fa2ca69a779b690b980e0bbcce3009fb93482...,formula-1-which-team-will-collect-more-points-...,https://www.formula1.com/en/racing/2022/Austri...,2022-07-10T00:00:00Z,Sports,110.248463,2022-07-08T19:18:22.58Z,5000000000000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,246671,Formula 1: Who will finish ahead at the 2022 A...,0x4a6068fd0d4bb6eee722fe137a2a66ac4fe96aecd244...,formula-1-who-will-finish-ahead-at-the-2022-au...,https://www.formula1.com/en/racing/2022/Austri...,2022-07-10T00:00:00Z,Sports,122.474506,2022-07-08T19:18:32.953Z,5000000000000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,246672,Formula 1: Will 17 or more drivers complete th...,0x36a5578179be0b36bcaf4124a543ca1356facce68180...,formula-1-will-17-or-more-drivers-complete-the...,https://www.formula1.com/en/racing/2022/Austri...,2022-07-10T00:00:00Z,Sports,144.115351,2022-07-08T19:18:42.536Z,5000000000000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,246788,Formula 1: Which team will collect more points...,0x323ceeee1ce2f78f17fd5c09f7d44a8f727ade2349ea...,formula-1-which-team-will-collect-more-points-...,https://www.formula1.com/en/racing/2022/France...,2022-07-24T00:00:00Z,Sports,83.369932,2022-07-22T17:57:40.591Z,5000000000000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,246789,Formula 1: Will 17 or more drivers complete th...,0x9e7ac31c0deedbbd7b7f25a6cd62d3cb539a1f3d356c...,formula-1-will-17-or-more-drivers-complete-the...,https://www.formula1.com/en/racing/2022/France...,2022-07-24T00:00:00Z,Sports,96.086521,2022-07-22T17:58:02.754Z,5000000000000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Inspect what Polymarket's "sports" returned

We inspect the returned markets to understand:

- how many markets were collected;
- which original `category` values appear;
- whether "miscategorized" sports markets are recovered.

A market can therefore appear in this dataset even if its old `category` field is something other than `Sports`, because the query is based on Polymarket's **tag taxonomy**, not solely the category column.


In [7]:
print("Total rows:", len(markets_df))

if "category" in markets_df.columns:
    print("\nOriginal category values:")
    display(
        markets_df["category"]
        .value_counts(dropna=False)
        .rename_axis("category")
        .reset_index(name="count")
    )


Total rows: 2557889

Original category values:


,category,count
0,NaN,2557879
1,Sports,10


### Check the Khabib example (he was misclassified as pop-culture category before)

> **Will Khabib win his UFC 254 Fight?**

Serves as a sanity check for whether the Sports-tag query catches sports markets that the old category field missed.


In [8]:
khabib_check = markets_df[
    markets_df["question"].str.contains(
        "Khabib",
        case=False,
        na=False,
    )
]

display_columns = [
    col for col in [
        "id",
        "question",
        "category",
        "endDate",
    ]
    if col in khabib_check.columns
]

khabib_check[display_columns]


,id,question,category,endDate
864794,2859751,"Will the announcers say ""Khabib"" or ""Oliveira""...",NaN,2026-07-12T23:59:00Z
864813,2859770,"Will the announcers say ""Khabib"" or ""Oliveira""...",NaN,2026-07-11T00:00:00Z
864839,2859796,"Will the announcers say ""Khabib"" or ""Oliveira""...",NaN,2026-07-12T23:59:00Z
1365780,3574329,"Will the announcers say ""Nurmagomedov"" or ""Kha...",NaN,2026-08-15T23:59:00Z
2122228,4643416,"Will the announcers say ""Nurmagomedov"" or ""Kha...",NaN,2026-09-19T23:59:00Z
2122242,4643430,"Will the announcers say ""Nurmagomedov"" or ""Kha...",NaN,2026-09-19T23:59:00Z


## 4. Build the dataset

At this stage we preserve the candidate sports markets rather than making further analytical decisions.

In particular:

- a row does **not** need to represent a head-to-head match yet;
- a row does **not** need to be binary yet;
- `outcomePrices` are retained as raw Polymarket metadata, not interpreted as labels;
- CLOB token IDs are retained for later historical-price work.

Stable identifiers such as `market_id` and `condition_id` will let later stages join annotations, resolved outcomes, and price histories back onto these rows.


In [9]:
sports_df = markets_df.copy()

# Convert useful time fields when present
for col in ["startDate", "endDate", "createdAt", "closedTime"]:
    if col in sports_df.columns:
        sports_df[col] = pd.to_datetime(
            sports_df[col],
            utc=True,
            errors="coerce",
        )

# Convert useful numeric metadata when present
for col in ["volumeNum", "liquidityNum"]:
    if col in sports_df.columns:
        sports_df[col] = pd.to_numeric(
            sports_df[col],
            errors="coerce",
        )


wanted_columns = [
    # Stable identifiers
    "id",
    "conditionId",
    "questionID",

    # Market text
    "question",
    "slug",
    "description",

    # Timing / status
    "startDate",
    "endDate",
    "createdAt",
    "closedTime",
    "closed",

    # Original Polymarket metadata
    "category",
    "subcategory",
    "tags",
    "events",

    # Outcomes / token metadata for later stages
    "outcomes",
    "outcomePrices",
    "clobTokenIds",

    # Market metadata
    "volumeNum",
    "liquidityNum",
    "resolutionSource",
]

available_columns = [
    col for col in wanted_columns
    if col in sports_df.columns
]

closed_sports_markets = sports_df[available_columns].copy()

closed_sports_markets = closed_sports_markets.rename(
    columns={
        "id": "market_id",
        "conditionId": "condition_id",
        "questionID": "question_id",
    }
)

closed_sports_markets.head()


,market_id,condition_id,question_id,question,slug,description,startDate,endDate,createdAt,closedTime,closed,category,events,outcomes,outcomePrices,clobTokenIds,volumeNum,liquidityNum,resolutionSource
0,246670,0x6818e44fa2ca69a779b690b980e0bbcce3009fb93482...,0x16742adb72e2d65eeb46a84061d8cb82cee91ae30d2e...,Formula 1: Which team will collect more points...,formula-1-which-team-will-collect-more-points-...,This is a market on which constructor team wil...,2022-07-08 19:18:22.580000+00:00,2022-07-10 00:00:00+00:00,2022-07-08 17:06:00.731000+00:00,2022-07-10 18:45:43+00:00,True,Sports,"[{'id': '5610', 'ticker': 'f1-2022austriangp',...","[""Red Bull"", ""Ferrari""]","[""0.000001558360149007031350273565876905504"", ...","[""14296772239362844550585638456903814361785758...",2239.21,110.25,https://www.formula1.com/en/racing/2022/Austri...
1,246671,0x4a6068fd0d4bb6eee722fe137a2a66ac4fe96aecd244...,0xdc46142c38a7161c053dad8aa122f47a41b283c86c94...,Formula 1: Who will finish ahead at the 2022 A...,formula-1-who-will-finish-ahead-at-the-2022-au...,This is a market on whether Max Verstappen or ...,2022-07-08 19:18:32.953000+00:00,2022-07-10 00:00:00+00:00,2022-07-08 17:11:59.224000+00:00,2022-07-10 18:46:01+00:00,True,Sports,"[{'id': '5610', 'ticker': 'f1-2022austriangp',...","[""Verstappen"", ""Leclerc""]","[""0.000000689240582043482038042890082149935"", ...","[""62081099646400111580527560634615779100657299...",2090.88,122.47,https://www.formula1.com/en/racing/2022/Austri...
2,246672,0x36a5578179be0b36bcaf4124a543ca1356facce68180...,0x440f651a9253aecd525906eb835cd1742d2d53220913...,Formula 1: Will 17 or more drivers complete th...,formula-1-will-17-or-more-drivers-complete-the...,The 2022 Austrian Grand Prix is a Formula One ...,2022-07-08 19:18:42.536000+00:00,2022-07-10 00:00:00+00:00,2022-07-08 17:22:08.463000+00:00,2022-07-10 19:00:53+00:00,True,Sports,"[{'id': '5610', 'ticker': 'f1-2022austriangp',...","[""Yes"", ""No""]","[""0.9999991948870350155081495065164764"", ""0.00...",NaN,1614.29,144.12,https://www.formula1.com/en/racing/2022/Austri...
3,246788,0x323ceeee1ce2f78f17fd5c09f7d44a8f727ade2349ea...,0x384971f1898c38379ac16eb0f28730499635e10d750e...,Formula 1: Which team will collect more points...,formula-1-which-team-will-collect-more-points-...,This is a market on which constructor team wil...,2022-07-22 17:57:40.591000+00:00,2022-07-24 00:00:00+00:00,2022-07-22 11:53:03.295000+00:00,2022-07-25 03:15:54+00:00,True,Sports,"[{'id': '5781', 'ticker': 'F1-frenchgp2022', '...","[""Red Bull"", ""Ferrari""]","[""0.9999993331626557600303201012307052"", ""0.00...","[""15664927994920424642777923591835423310456470...",3747.09,83.37,https://www.formula1.com/en/racing/2022/France...
4,246789,0x9e7ac31c0deedbbd7b7f25a6cd62d3cb539a1f3d356c...,0xb85360e23e59892e9344d0d28a5390d067a450d38e75...,Formula 1: Will 17 or more drivers complete th...,formula-1-will-17-or-more-drivers-complete-the...,The 2022 French Grand Prix is a Formula One mo...,2022-07-22 17:58:02.754000+00:00,2022-07-24 00:00:00+00:00,2022-07-22 11:56:19.429000+00:00,2022-07-25 03:15:36+00:00,True,Sports,"[{'id': '5781', 'ticker': 'F1-frenchgp2022', '...","[""Yes"", ""No""]","[""0.00000142418528179466515776028054868592"", ""...","[""56238254368341581410666130055984373644338613...",1201.67,96.09,https://www.formula1.com/en/racing/2022/France...


## 5. Basic dataset checks

These checks are only about collection quality. We are not yet deciding whether a market is a suitable match-level observation for the eventual analysis.


In [10]:
print("Rows:", len(closed_sports_markets))

if "market_id" in closed_sports_markets.columns:
    print(
        "Duplicate market IDs:",
        closed_sports_markets["market_id"].duplicated().sum(),
    )

if "closed" in closed_sports_markets.columns:
    print(
        "Rows not marked closed:",
        (~closed_sports_markets["closed"].fillna(False)).sum(),
    )

if "category" in closed_sports_markets.columns:
    print("\nOriginal categories:")
    print(
        closed_sports_markets["category"]
        .value_counts(dropna=False)
        .head(20)
    )


Rows: 2557889
Duplicate market IDs: 0
Rows not marked closed: 0

Original categories:
category
NaN       2557879
Sports         10
Name: count, dtype: int64


In [11]:
preview_columns = [
    col for col in [
        "market_id",
        "question",
        "category",
        "endDate",
        "outcomes",
    ]
    if col in closed_sports_markets.columns
]

closed_sports_markets[preview_columns].head(50)


,market_id,question,category,endDate,outcomes
0,246670,Formula 1: Which team will collect more points...,Sports,2022-07-10 00:00:00+00:00,"[""Red Bull"", ""Ferrari""]"
1,246671,Formula 1: Who will finish ahead at the 2022 A...,Sports,2022-07-10 00:00:00+00:00,"[""Verstappen"", ""Leclerc""]"
2,246672,Formula 1: Will 17 or more drivers complete th...,Sports,2022-07-10 00:00:00+00:00,"[""Yes"", ""No""]"
3,246788,Formula 1: Which team will collect more points...,Sports,2022-07-24 00:00:00+00:00,"[""Red Bull"", ""Ferrari""]"
4,246789,Formula 1: Will 17 or more drivers complete th...,Sports,2022-07-24 00:00:00+00:00,"[""Yes"", ""No""]"
5,246790,Formula 1: Who will finish ahead at the 2022 F...,Sports,2022-07-24 00:00:00+00:00,"[""Verstappen"", ""Leclerc""]"
6,246856,Formula 1: Who will finish ahead at the 2022 H...,Sports,2022-07-31 00:00:00+00:00,"[""Verstappen"", ""Leclerc""]"
7,247139,Formula 1: Which team will collect more points...,Sports,2022-08-28 00:00:00+00:00,"[""Mercedes"", ""Ferrari""]"
8,247140,Formula 1: Will Max Verstappen win the 2022 Be...,Sports,2022-08-28 00:00:00+00:00,"[""Yes"", ""No""]"
9,247141,Formula 1: Will 17 or more drivers complete th...,Sports,2022-08-28 00:00:00+00:00,"[""Yes"", ""No""]"


## 6. Save the Task 1 dataset

This notebook produces one dataset:

```text
data/closed_sports_markets.csv
```

The next project stages can then work from this fixed candidate universe:

```text
closed sports candidates
        ↓
resolved outcomes / labels
        ↓
manual annotation of actual sports matches
        ↓
historical prices
        ↓
analysis
```


In [12]:
OUTPUT_PATH = DATA_DIR / "closed_sports_markets.csv"

closed_sports_markets.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Saved {len(closed_sports_markets)} closed sports-market candidates")
print(OUTPUT_PATH)


Saved 2557889 closed sports-market candidates
data\closed_sports_markets.csv


# Notes on the selection rule

The selection rule for this dataset is intentionally simple and reproducible:

> Include a market if Gamma returns it for  
> `closed=true`, `tag_id=Sports`, `related_tags=true`.

We do **not** supplement that result with our own sports keyword list.

This means Polymarket's tag taxonomy defines the initial candidates. Any later questions such as:

- "Is this really a sports market?"
- "Is this an individual match?"
- "Which sport / league is this?"
- "Should this row be included in the final analysis?"

belongs to the later **manual annotation** stage rather than this collection notebook.
